# FloodPredicta (Version 2.0) - Data Preprocessing & Cleaning

This notebook performs data auditing, target missing value imputation, extreme value analysis, log transformations, and sorting for the **FloodPredicta** flood prediction system.

### Key Objectives in Version 2.0:
1. Audit raw meteorological observations across 33 Bangladesh Meteorological Department (BMD) stations (1948–2013).
2. Resolve target variable `Flood?` missing values with domain-backed justification.
3. **Preserve extreme weather phenomena:** Explicitly demonstrate why aggressive 5-pass IQR outlier dropping (used in Version 1) deletes thousands of genuine flood disaster cases, and adopt non-destructive winsorization instead.
4. Correct severe right-skewness using $\ln(1 + x)$ logarithmic transformations.
5. Chronologically sort station records to prepare for temporal and hydrological lag feature engineering.
6. Export the clean dataset to `data/processed/flood_data_preprocessed.csv`.


In [1]:
# Import core libraries for data processing and analysis
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean pandas display formatting
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version:  {np.__version__}")


Libraries imported successfully.
Pandas version: 2.3.2
Numpy version:  2.3.3



## 1. Load Raw Climate Dataset
Load the raw dataset from `../data/raw/flooddata.csv` and inspect its initial shape and structure.


In [2]:
# Load raw meteorological dataset
raw_data_path = os.path.join('..', 'data', 'raw', 'flooddata.csv')
df = pd.read_csv(raw_data_path)

print(f"Raw dataset loaded successfully from: {raw_data_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Raw dataset loaded successfully from: ..\data\raw\flooddata.csv
Shape: 20,544 rows x 19 columns



   Sl Station_Names  Year  Month  Max_Temp  Min_Temp  Rainfall  Relative_Humidity  Wind_Speed  Cloud_Coverage  Bright_Sunshine  Station_Number      X_COR      Y_COR  LATITUDE  LONGITUDE  ALT   Period  Flood?
0   0       Barisal  1949      1    29.400    12.300     0.000             68.000       0.454           0.600            7.832           41950 536809.800 510151.900    22.700     90.360    4 1949.010     NaN
1   1       Barisal  1949      2    33.900    15.200     9.000             63.000       0.659           0.900            8.315           41950 536809.800 510151.900    22.700     90.360    4 1949.020     NaN
2   2       Barisal  1949      3    36.700    20.200     8.000             59.000       1.085           1.500            8.132           41950 536809.800 510151.900    22.700     90.360    4 1949.030     NaN
3   3       Barisal  1949      4    33.900    23.900   140.000             71.000       1.772           3.900            8.219           41950 536809.800 510151.900    

## 2. Inspect Column Types and Missing Values
Check data types across all 19 columns and identify where missing values exist.


In [3]:
# Check data types and identify missing values
print("--- Dataset Information ---")
df.info()

print("\n--- Missing Values Audit ---")
missing = df.isna().sum()
print(missing[missing > 0])
print(f"\nConclusion: Only 'Flood?' has missing entries ({missing['Flood?']:,} rows). All 18 meteorological and geographic features are complete.")


--- Dataset Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20544 entries, 0 to 20543
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Sl                 20544 non-null  int64  
 1   Station_Names      20544 non-null  object 
 2   Year               20544 non-null  int64  
 3   Month              20544 non-null  int64  
 4   Max_Temp           20544 non-null  float64
 5   Min_Temp           20544 non-null  float64
 6   Rainfall           20544 non-null  float64
 7   Relative_Humidity  20544 non-null  float64
 8   Wind_Speed         20544 non-null  float64
 9   Cloud_Coverage     20544 non-null  float64
 10  Bright_Sunshine    20544 non-null  float64
 11  Station_Number     20544 non-null  int64  
 12  X_COR              20544 non-null  float64
 13  Y_COR              20544 non-null  float64
 14  LATITUDE           20544 non-null  float64
 15  LONGITUDE          20544 non-null  float64

## 3. Target Variable Analysis & Missing Value Imputation
In the raw data, `Flood?` contains 16,051 `NaN` entries, 4,132 `1.0` flags, and 361 `0.0` flags.
In meteorological and disaster archiving in Bangladesh, actual flood events were noted (`1.0`), while normal non-flood months were left unrecorded (`NaN`).
Therefore, `NaN` values are imputed with `0.0`.


In [4]:
# Check target variable value counts before and after imputation
print("Target 'Flood?' counts before imputation:")
print(df['Flood?'].value_counts(dropna=False))

# Impute NaN -> 0.0 based on domain record keeping convention
df['Flood?'] = df['Flood?'].fillna(0.0)

print("\nTarget 'Flood?' counts after imputation:")
counts = df['Flood?'].value_counts()
print(counts)
print(f"\nClass 0.0 (Non-Flood): {counts[0.0]:,} ({counts[0.0]/len(df)*100:.2f}%)")
print(f"Class 1.0 (Flood):     {counts[1.0]:,} ({counts[1.0]/len(df)*100:.2f}%)")
print(f"Imbalance Ratio:       ~{counts[0.0]/counts[1.0]:.2f} : 1")


Target 'Flood?' counts before imputation:
Flood?
NaN      16051
1.000     4132
0.000      361
Name: count, dtype: int64

Target 'Flood?' counts after imputation:
Flood?
0.000    16412
1.000     4132
Name: count, dtype: int64

Class 0.0 (Non-Flood): 16,412 (79.89%)
Class 1.0 (Flood):     4,132 (20.11%)
Imbalance Ratio:       ~3.97 : 1



## 4. Meteorological Station Coverage & Timeline
Examine the monitored weather stations and the overall historical time span.


In [5]:
# Unique meteorological stations and observation years
stations = df['Station_Names'].unique()
print(f"Total Unique Weather Stations: {len(stations)}")
print(f"Stations list: {sorted(stations)}")

min_yr, max_yr = df['Year'].min(), df['Year'].max()
print(f"\nObservation Timeline: {min_yr} to {max_yr} ({max_yr - min_yr + 1} years)")
print(f"Average observations per station: {int(df['Station_Names'].value_counts().mean())} months")


Total Unique Weather Stations: 33
Stations list: ['Barisal', 'Bhola', 'Bogra', 'Chandpur', 'Chittagong (City-Ambagan)', 'Chittagong (IAP-Patenga)', 'Comilla', "Cox's Bazar", 'Dhaka', 'Dinajpur', 'Faridpur', 'Feni', 'Hatiya', 'Ishurdi', 'Jessore', 'Khepupara', 'Khulna', 'Kutubdia', 'Madaripur', 'Maijdee Court', 'Mongla', 'Mymensingh', 'Patuakhali', 'Rajshahi', 'Rangamati', 'Rangpur', 'Sandwip', 'Satkhira', 'Sitakunda', 'Srimangal', 'Sylhet', 'Tangail', 'Teknaf']

Observation Timeline: 1948 to 2013 (66 years)
Average observations per station: 622 months



## 5. Remove Redundant Identifier Columns
- `Station_Number` is completely redundant with `Station_Names`.
- `Sl` is an arbitrary sequence counter with no predictive signal.
We drop both to keep our feature matrix clean.


In [6]:
# Drop redundant identifiers
cols_to_drop = ['Sl', 'Station_Number']
existing_cols = [c for c in cols_to_drop if c in df.columns]

df.drop(columns=existing_cols, inplace=True)
print(f"Dropped redundant columns: {existing_cols}")
print(f"Updated DataFrame shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


Dropped redundant columns: ['Sl', 'Station_Number']
Updated DataFrame shape: 20,544 rows x 17 columns



## 6. Summary Statistics of Numeric Features
Review the central tendency, dispersion, and spread of all weather variables.


In [7]:
# Numerical summary statistics
df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]


                      count       mean        std      min        25%        50%        75%        max
Year              20544.000   1985.333     17.611 1948.000   1972.000   1987.000   2000.000   2013.000
Month             20544.000      6.500      3.452    1.000      3.750      6.500      9.250     12.000
Max_Temp          20544.000     33.451      2.956   21.600     31.700     33.900     35.400     44.000
Min_Temp          20544.000     21.167      4.950    6.200     16.900     23.400     25.400     28.100
Rainfall          20544.000    198.777    240.693    0.000      8.000    111.000    312.000   2072.000
Relative_Humidity 20544.000     79.497      7.668   34.000     75.000     81.000     85.000     97.000
Wind_Speed        20544.000      1.415      1.042    0.000      0.700      1.200      1.900     11.200
Cloud_Coverage    20544.000      3.486      2.084    0.000      1.600      3.300      5.500      7.900
Bright_Sunshine   20544.000      6.419      1.748    0.000      4.966    

## 7. Outlier & Extreme Event Analysis: Why V1 Was Flawed
In Version 1, 5 passes of IQR trimming were run on `Rainfall`, `Wind_Speed`, and temperature metrics.
Let's measure what IQR trimming actually does to our real historical flood records.


In [8]:
# Diagnostic: Measuring real flood disaster events lost to IQR trimming
features_to_check = ['Max_Temp', 'Rainfall', 'Relative_Humidity', 'Wind_Speed', 'ALT']

df_iqr_test = df.copy()
initial_floods = int((df_iqr_test['Flood?'] == 1.0).sum())

# Run 1 pass of IQR filtering
for col in features_to_check:
    q1 = df_iqr_test[col].quantile(0.25)
    q3 = df_iqr_test[col].quantile(0.75)
    iqr = q3 - q1
    lower_lim = q1 - 1.5 * iqr
    upper_lim = q3 + 1.5 * iqr
    df_iqr_test = df_iqr_test[(df_iqr_test[col] >= lower_lim) & (df_iqr_test[col] <= upper_lim)]

remaining_floods = int((df_iqr_test['Flood?'] == 1.0).sum())
lost_floods = initial_floods - remaining_floods

print("=== CRITICAL OUTLIER ANALYSIS ===")
print(f"Initial Verified Flood Events:                 {initial_floods:,}")
print(f"Flood Events Remaining After 1 IQR Pass:       {remaining_floods:,}")
print(f"Genuine Flood Disasters Deleted by IQR:        {lost_floods:,} ({lost_floods/initial_floods*100:.2f}% of all floods!)")
print("\nDECISION: In Version 2.0, we DO NOT drop rows using IQR.")
print("Floods are physically driven by extreme weather. Trimming extreme rain discards the very disasters we must predict.")


=== CRITICAL OUTLIER ANALYSIS ===
Initial Verified Flood Events:                 4,132
Flood Events Remaining After 1 IQR Pass:       3,102
Genuine Flood Disasters Deleted by IQR:        1,030 (24.93% of all floods!)

DECISION: In Version 2.0, we DO NOT drop rows using IQR.
Floods are physically driven by extreme weather. Trimming extreme rain discards the very disasters we must predict.



## 8. Non-Destructive Quantile Capping (Winsorization)
To prevent impossible physical sensor anomalies from distorting calculations while preserving 99.5% of real monsoon deluges, we apply conservative 0.5% lower and 99.5% upper quantile capping.


In [9]:
# Apply conservative winsorization (capping)
capping_cols = ['Rainfall', 'Wind_Speed', 'Max_Temp', 'Relative_Humidity']

print("Applied Capping Bounds (0.5% lower, 99.5% upper):")
for col in capping_cols:
    lower_bound = df[col].quantile(0.005)
    upper_bound = df[col].quantile(0.995)
    prev_max = df[col].max()
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    print(f"  - {col:18s}: lower={lower_bound:8.2f}, upper={upper_bound:8.2f} (prev max={prev_max:8.2f})")

print(f"\nPreserved all {len(df):,} observations without losing flood instances!")


Applied Capping Bounds (0.5% lower, 99.5% upper):
  - Rainfall          : lower=    0.00, upper= 1201.57 (prev max= 2072.00)
  - Wind_Speed        : lower=    0.00, upper=    5.60 (prev max=   11.20)
  - Max_Temp          : lower=   26.00, upper=   41.23 (prev max=   44.00)
  - Relative_Humidity : lower=   53.00, upper=   92.00 (prev max=   97.00)

Preserved all 20,544 observations without losing flood instances!



## 9. Skewness Analysis & Logarithmic Transformations
Precipitation and wind velocity exhibit extreme right-skewed tails.
Applying $\ln(1 + x)$ transforms compress these long tails, enabling stable distance calculations and linear relationships.


In [10]:
# Assess skewness and apply log1p transforms
numeric_features = df.select_dtypes(include=['number']).columns.drop('Flood?')
print("--- Skewness Values Before Log Transformation ---")
skew_before = df[numeric_features].skew().sort_values(ascending=False)
print(skew_before)

# Apply log1p on heavily right-skewed features
skewed_cols = ['Rainfall', 'Wind_Speed', 'ALT']

print("\n--- Applying log1p Transformations ---")
for col in skewed_cols:
    log_name = col + ' log'
    df[log_name] = np.log1p(df[col])
    print(f"Created: {log_name:15s} (Skewness improved: {df[col].skew():.3f} -> {df[log_name].skew():.3f})")


--- Skewness Values Before Log Transformation ---
ALT                  1.999
Rainfall             1.586
Wind_Speed           1.371
LATITUDE             0.361
Y_COR                0.159
Cloud_Coverage       0.107
Month                0.000
LONGITUDE           -0.067
X_COR               -0.270
Period              -0.275
Year                -0.275
Max_Temp            -0.325
Bright_Sunshine     -0.335
Min_Temp            -0.720
Relative_Humidity   -0.888
dtype: float64

--- Applying log1p Transformations ---
Created: Rainfall log    (Skewness improved: 1.586 -> -0.645)
Created: Wind_Speed log  (Skewness improved: 1.371 -> 0.337)
Created: ALT log         (Skewness improved: 1.999 -> 0.695)



## 10. Chronological Sorting by Station and Time
For temporal consistency and future lag feature engineering (such as previous month rainfall and cumulative monsoonal saturation),
we ensure records are strictly sorted by `Station_Names`, `Year`, and `Month`.


In [11]:
# Chronological sort per meteorological station
df.sort_values(by=['Station_Names', 'Year', 'Month'], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Chronological sequence established. First 5 records of first station:")
df[['Station_Names', 'Year', 'Month', 'Period', 'Rainfall', 'Rainfall log', 'Flood?']].head()


Chronological sequence established. First 5 records of first station:



  Station_Names  Year  Month   Period  Rainfall  Rainfall log  Flood?
0       Barisal  1949      1 1949.010     0.000         0.000   0.000
1       Barisal  1949      2 1949.020     9.000         2.303   0.000
2       Barisal  1949      3 1949.030     8.000         2.197   0.000
3       Barisal  1949      4 1949.040   140.000         4.949   0.000
4       Barisal  1949      5 1949.050   217.000         5.384   0.000


## 11. Final Dataset Verification & Export
Verify complete absence of missing values, check final record counts, and save the preprocessed dataset to `version2/data/processed/flood_data_preprocessed.csv`.


In [12]:
# Final verification and export
print("=== PREPROCESSED DATASET QUALITY REPORT ===")
print(f"Total Rows:           {df.shape[0]:,}")
print(f"Total Columns:        {df.shape[1]}")
print(f"Missing Values:       {df.isna().sum().sum()}")
print(f"Final Class Counts:   Non-Flood={int((df['Flood?']==0).sum()):,}, Flood={int((df['Flood?']==1).sum()):,}")
print(f"Columns in dataset:   {list(df.columns)}")

processed_dir = os.path.join('..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
output_path = os.path.join(processed_dir, 'flood_data_preprocessed.csv')

df.to_csv(output_path, index=False)
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nSuccessfully exported preprocessed dataset to: {output_path}")
print(f"File size on disk: {file_size_mb:.2f} MB")


=== PREPROCESSED DATASET QUALITY REPORT ===
Total Rows:           20,544
Total Columns:        20
Missing Values:       0
Final Class Counts:   Non-Flood=16,412, Flood=4,132
Columns in dataset:   ['Station_Names', 'Year', 'Month', 'Max_Temp', 'Min_Temp', 'Rainfall', 'Relative_Humidity', 'Wind_Speed', 'Cloud_Coverage', 'Bright_Sunshine', 'X_COR', 'Y_COR', 'LATITUDE', 'LONGITUDE', 'ALT', 'Period', 'Flood?', 'Rainfall log', 'Wind_Speed log', 'ALT log']

Successfully exported preprocessed dataset to: ..\data\processed\flood_data_preprocessed.csv
File size on disk: 2.96 MB

